In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
# Just tri twice

import gin
import numpy as np
import torch
import pytorch_lightning
from deepgene.models import GruLabeler, LstmLabeler
from deepgene.data import DatasetXY
from tqdm import tqdm
import scipy 
import seaborn as sns
from collections import Counter

from deepgene.data import data_gen_np as dgn

from matplotlib import pyplot as plt
from os.path import join

sns.set() # Setting seaborn as default style even if use only matplotlib
sns.set_context("paper")

/home/evkhomutov/.conda/envs/my_py_env/lib/python3.8/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import gin
gin.clear_config()
gin.parse_config_files_and_bindings(
    [
        "configs/gru/data2.gin", 
        "configs/gru/train.gin", 
        "configs/gru/model.gin"
    ], "")

[ParsedConfigFileIncludesAndImports(filename='configs/gru/data2.gin', imports=['deepgene.data'], includes=[]),
 ParsedConfigFileIncludesAndImports(filename='configs/gru/train.gin', imports=[], includes=[]),
 ParsedConfigFileIncludesAndImports(filename='configs/gru/model.gin', imports=['deepgene.models'], includes=[])]

# Model

In [4]:
gru_model = GruLabeler.load_from_checkpoint(checkpoint_path="models_weights/gru_5h_ol_hl.ckpt").eval()

In [8]:
gru_model.device

device(type='cuda', index=0)

# Data

In [9]:
def give_true_predicted(model, dem, lengt=3_000_000, num_gens=1):

    lengt = lengt * num_gens
    dg = dgn.DataGenerator(
        lengt=lengt,
        num_replicates=num_gens,
        demographic_events= dem,
        genome_postproccessor=dgn.do_filter,
        times_postproccessor=dgn.do_filter_2
    ) ## -> (binary_genome, prior_dist, local_coalescnet_time)

    data = [gen for gen in dg]
    X = torch.tensor([data[0][0]])
    y_true = [data[0][2]]

    y_pred = model(X).detach().numpy()
    return y_true, y_pred

In [ ]:
%%time
y_true, y_pred = give_true_predicted(gru_model, dgn.get_humanlike(), num_gens=100)

In [ ]:
# data = DatasetXY().test_dataloader()

# data_point_1 = next(data)

#X = data_point_1[0]
#y_true = data_point_1[1]

# Pred

In [ ]:
# y_pred = gru_model(X).detach().numpy()

In [ ]:
def plot_heatmap(y_pred, y_true, length):
    sns.set(rc={'figure.figsize' : (11.7, 8.27),
           'axes.labelsize': 20,
           'axes.titlesize': 20, 
            'figure.dpi':100
           }
       )
    ax = sns.heatmap(y_pred.squeeze(0)[:length].T, cmap="viridis", yticklabels=5, xticklabels=1000)
    ax.plot(y_true[0][:length], c='red')
    _ = ax.set(xlabel='Genome wide', ylabel='Coalescent time')
    #ax.set_ylim([23, 0])
    plt.show()

In [ ]:
def myexp(data):
    return scipy.special.softmax(data, axis=-1)

In [ ]:
plot_heatmap(myexp(y_pred), y_true, 3000)

In [ ]:
def f(x):
    return myexp(x[0]).argmax(axis=1)

def NN2cdf(X, num_intervals = 32):
    cnt = Counter(X)
    pd = [0 for i in range(num_intervals)]
    for (key, value) in cnt.items(): pd[key] = value
    pd = np.array(pd) / sum(pd)
    return [sum(pd[:i]) for i in range(len(pd))]

def cdf2NE(cdf_values, limits=dgn.limits, posinf=10_000, neginf=10_000):
    # Estimate Ne from the CDF
    ne_estimates = []
    for t, c in zip(limits, cdf_values):
        if c < 1.0:  # Avoid log(0)
            ne = -t / (2 * np.log(1 - c))
            ne_estimates.append(ne)
        else:
            ne_estimates.append(ne)
            #ne_estimates.append(np.nan)  # For c=1, Ne is undefined (or infinite)

    # return ne_estimates
    return np.nan_to_num(ne_estimates, posinf=posinf, neginf=neginf)


const_nn_const = NN2cdf(f(y_pred))
Ne_const_nn_const = cdf2NE(const_nn_const)

In [ ]:
const_nn_const

In [ ]:
def dem2Ne(demography, limits=dgn.limits):
    debug = demography.debug()
    return debug.population_size_trajectory(limits)

plt.step(dgn.limits, dem2Ne(dgn.get_humanlike()), label="True", where='post')
plt.step(dgn.limits, Ne_const_nn_const, label="Predicted", where='post')

plt.xlabel("Time ago (in generation)")
plt.ylabel("Population size")
plt.xscale("log")

plt.legend()

In [ ]:
1 + 1

In [ ]:
def dem2Ne(demography, limits=dgn.limits):
    debug = demography.debug()
    return debug.population_size_trajectory(limits)

plt.step(dgn.limits, dem2Ne(dgn.get_humanlike()), label="Predicted", where='post')
plt.xlabel("Time ago")
plt.ylabel("Population size")
plt.xscale("log")

In [28]:
from scipy.integrate import quad

def give_cdf(times, Ne):
    def N_e(t, times_real = times):
        conditions = [(t < times_real[0])]
        conditions += [(t >= times_real[i]) & (t < times_real[i + 1]) for i in range(len(times_real) - 1)]
        conditions += [(t >= times_real[-1])]
        functions = [Ne[0]] + list(Ne) + [Ne[-1]]
        return np.piecewise(t, conditions, functions)

    def coalescent_rate(t):
        return 1 / (2 * N_e(t))
    
    # Cumulative hazard function using numerical integration
    def cumulative_hazard(t):
        result, _ = quad(coalescent_rate, 0, t)
        return result
    
    # Define a range of t values
    # t_values = np.linspace(0, max(x_values), 1000)
    
    # Compute the cumulative hazard function for each t
    Lambda_t = np.array([cumulative_hazard(t) for t in times])
    
    # Compute the CDF of the coalescent time
    return 1 - np.exp(-Lambda_t)

true_cdf = give_cdf(dgn.limits, dem2Ne(dgn.get_humanlike()))

/var/folders/6r/nbxt9z1x3sz2dbv5fyvkf2hh0000gn/T/ipykernel_14177/2721731373.py:16: IntegrationWarning: The maximum number of subdivisions (50) has been achieved.
  If increasing the limit yields no improvement it is advised to analyze 
  the integrand in order to determine the difficulties.  If the position of a 
  local difficulty can be determined (singularity, discontinuity) one will 
  probably gain from splitting up the interval and calling the integrator 
  on the subranges.  Perhaps a special-purpose integrator should be used.
  result, _ = quad(coalescent_rate, 0, t)


In [29]:
for x,y in zip(const_nn_const, true_cdf):
    print(f"{x} vs {y}")

0 vs 0.0
0.0 vs 0.010184908420157734
0.0 vs 0.023992614163599657
0.0 vs 0.04262739576566155
0.0 vs 0.06762140972452813
0.004563333333333333 vs 0.10086104635897386
0.009256666666666667 vs 0.14455422006277774
0.009256666666666667 vs 0.2010776227611495
0.022539999999999998 vs 0.687313165807129
0.03164333333333333 vs 0.9136906095525315
0.045989999999999996 vs 0.9852465228195711
0.07293666666666666 vs 0.9986933870467533
0.11832666666666666 vs 0.9999530641634112
0.17593666666666666 vs 0.9999995112568976
0.3719366666666667 vs 0.9999997387593813
0.62782 vs 0.9999998894060514
0.78705 vs 0.99999996600193
0.8952866666666667 vs 0.9999999986648986
0.9482666666666667 vs 0.999999999984288
0.9692700000000001 vs 0.9999999999999646
0.9889166666666668 vs 1.0
0.9949833333333334 vs 1.0
0.9978633333333334 vs 1.0
0.9989333333333335 vs 1.0
0.9994933333333335 vs 1.0
0.9997733333333334 vs 1.0
0.99982 vs 1.0
0.99995 vs 1.0
1.0 vs 1.0
1.0 vs 1.0
1.0 vs 1.0
1.0 vs 1.0


In [33]:
N = 32  # int(sys.argv[4])

T_max = 20  # 400_000
coeff = np.log(T_max) / (N - 1)
MAGIC_COEFF = 550 # 2_000 ## 1_000 for 169302 as T_max, 2_000 for 338605
cut_coeff = 400.0

limits = [np.exp(coeff * i) for i in range(N)]
limits = [MAGIC_COEFF * (np.exp(i * np.log(1 + 10 * T_max) / N) - 1)
          for i in range(N)]

limits

[0.0,
 99.1387969342603,
 216.1475957913797,
 354.2475065565313,
 517.2402518852934,
 709.6128239066114,
 936.66100566128,
 1204.6351575709834,
 1520.9122822621537,
 1894.1991044800684,
 2334.771756636342,
 2854.7586682415417,
 3468.474446825107,
 4192.813941678395,
 5047.717338516637,
 6056.719088550434,
 7247.595783316804,
 8653.13081047624,
 10312.016840620308,
 12269.92098945444,
 14580.741977975867,
 17308.093898738116,
 20527.057434483482,
 24326.246738063484,
 28810.248872338816,
 34102.50296487364,
 40348.698337052636,
 47720.785153984674,
 56421.708003508684,
 66690.99271397748,
 78811.34020912125,
 93116.40892079906]

In [31]:
N = 32  # int(sys.argv[4])

T_max = 2500  # 400_000
coeff = np.log(T_max) / (N - 1)
MAGIC_COEFF = 550 # 2_000 ## 1_000 for 169302 as T_max, 2_000 for 338605
cut_coeff = 400.0

limits = [np.exp(coeff * i) for i in range(N)]
limits = [MAGIC_COEFF * (np.exp(i * np.log(1 + 10 * T_max) / N) - 1)
          for i in range(N)]

MAGIC_COEFF/25

22.0

In [36]:
def get_humanlike(initial_size: int = 10_000, random_seed: int = 42) -> 'msprime.Demography':
    
    # ms = "-eN 0.1 0.1 -eN 0.6 1 -eN  2 0.5 -eN 10 1 -eN 20 2"
    # graph = demes.from_ms(ms, N0=population, deme_names=["A"])
    # emography = msprime.Demography.from_demes(graph)

    demography = msprime.Demography()

    demography.add_population(name="A", initial_size=initial_size)
    demography.add_population_parameters_change(4000, initial_size=0.1 * initial_size)
    demography.add_population_parameters_change(24000, initial_size=1. * initial_size)
    demography.add_population_parameters_change(80000, initial_size=.5 * initial_size)
    demography.add_population_parameters_change(400000, initial_size=1. * initial_size)
    demography.add_population_parameters_change(800000, initial_size=2. * initial_size)

    return demography


0.93116